# Figure 2: Multi-Dataset Core Benchmark

- Figure panels in the manuscript: `Fig2a` to `Fig2e`
- Notebook outputs: standalone main-text PNGs for unified all-model benchmark panels, one shared legend, and one representative single-perturbation case plot
- Inputs: `artifacts/results/<dataset>/metrics_*.csv`, `artifacts/results/<baseline>/<dataset>/metrics.csv`, payload `pkl`, raw dataset `h5ad`
- Outputs: `artifacts/paper_figures/main/Fig2_MultiDatasetBenchmark/*`
- Run order: run after the core benchmark results for the selected single-perturbation datasets are ready
- Default datasets: `adamson`, `norman`, `dixit`
- Role: Main text benchmark only


In [1]:
from __future__ import annotations

import importlib
import sys
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from scripts.common.paper_plot_style import apply_gears_paper_style, style_axis as paper_style_axis
from scripts.trishift.analysis import baseline_panel

apply_gears_paper_style(font_scale=1.0)
from scripts.trishift.analysis._result_adapter import load_payload_item
from trishift._utils import load_adata, load_yaml, normalize_condition
importlib.reload(baseline_panel)


<module 'scripts.trishift.analysis.baseline_panel' from 'E:\\CODE\\trishift\\scripts\\trishift\\analysis\\baseline_panel.py'>

## Plan

- `Fig2a`: unified all-model `mean_pearson`
- `Fig2b`: unified all-model `mean_nmse`
- `Fig2c`: unified all-model `mean_deg_mean_r2`
- `Fig2d`: unified all-model `mean_systema_corr_20de_allpert` (temporary benchmark-side export; may later migrate fully to `Fig3`)
- `Fig2e`: one representative single-perturbation case plot for qualitative support
- Shared legend is exported as a standalone figure for the benchmark bar charts
- Systema/scPRAM and other mechanism-oriented panels are handled in `Fig3` or supplementary figures, not here
- Shared-baseline case is selected automatically from shared conditions using the displayed 12-gene direction-aware expression profile (pred - control)


In [2]:
DATASET_SPECS = [
    {'dataset_key': 'adamson', 'dataset_label': 'Adamson', 'benchmark_subgroup_filter': None},
    {'dataset_key': 'norman', 'dataset_label': 'Norman', 'benchmark_subgroup_filter': 'single'},
    {'dataset_key': 'dixit', 'dataset_label': 'Dixit', 'benchmark_subgroup_filter': None},
]
DATASET_ORDER = ['Adamson', 'Norman', 'Dixit']
MAIN_MODELS = ['trishift_nearest', 'gears', 'biolord', 'genepert', 'scgpt', 'systema_nonctl_mean']
AVAILABLE_SPLIT_IDS = [1, 2, 3, 4, 5]
SPLIT_IDS = AVAILABLE_SPLIT_IDS.copy()
PATHS_CFG = load_yaml(str(repo_root / 'configs' / 'paths.yaml'))

CASE_SHARED_BASELINE = {
    'dataset': 'adamson',
    'condition': None,
    'display_models': ['trishift_nearest', 'biolord', 'gears', 'genepert', 'scgpt'],
    'check_models': ['trishift_nearest', 'biolord', 'gears', 'genepert', 'scgpt'],
    'top_k': 12,
    'split_policy': 'first_available',
    'selection_policy': 'display_gene_expression_advantage',
}
MODEL_LABELS = {
    'trishift_nearest': 'TriShift',
    'biolord': 'biolord',
    'gears': 'GEARS',
    'genepert': 'GenePert',
    'scgpt': 'scGPT',
    'systema_nonctl_mean': 'Perturbed mean',
    'Truth': 'Truth',
}
MODEL_COLORS = {
    'TriShift': '#9FD9D3',
    'biolord': '#F0806A',
    'GEARS': '#F2B56B',
    'GenePert': '#87A8D8',
    'scGPT': '#DDD3C8',
    'Perturbed mean': '#B9AEDC',
    'Truth': '#7F7F7F',
}
OUT_ROOT = repo_root / 'artifacts' / 'paper_figures' / 'main' / 'Fig2_MultiDatasetBenchmark'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
for pattern in ['fig2*.png', 'fig2*.csv']:
    for stale_path in OUT_ROOT.glob(pattern):
        stale_path.unlink()

print('Selected datasets:', [spec['dataset_key'] for spec in DATASET_SPECS])
print('Selected splits:', SPLIT_IDS)
print('Shared-baseline case:', CASE_SHARED_BASELINE)


Selected datasets: ['adamson', 'norman', 'dixit']
Selected splits: [1, 2, 3, 4, 5]
Shared-baseline case: {'dataset': 'adamson', 'condition': None, 'display_models': ['trishift_nearest', 'biolord', 'gears', 'genepert', 'scgpt'], 'check_models': ['trishift_nearest', 'biolord', 'gears', 'genepert', 'scgpt'], 'top_k': 12, 'split_policy': 'first_available', 'selection_policy': 'display_gene_expression_advantage'}


In [3]:
def safe_run_baseline(dataset_key: str, models: list[str], tag: str, subgroup_filter: str | None = None) -> dict[str, object]:
    out_dir = OUT_ROOT / f'{tag}_{dataset_key}'
    out_dir.mkdir(parents=True, exist_ok=True)
    try:
        result = baseline_panel.run_baseline_panel(dataset=dataset_key, models=models, split_ids=SPLIT_IDS, subgroup_filter=subgroup_filter, out_root=out_dir)
        return {'status': 'ready', 'result': result, 'out_dir': out_dir}
    except Exception as exc:
        return {'status': 'pending', 'error': str(exc), 'out_dir': out_dir}


def _style_axis(ax: plt.Axes, *, grid_axis: str = 'y') -> None:
    paper_style_axis(ax, grid_axis=grid_axis)
    ax.set_axisbelow(True)


def _shrink_bars(ax: plt.Axes, scale: float = 0.9) -> None:
    for patch in ax.patches:
        width = patch.get_width()
        if width <= 0:
            continue
        new_width = width * scale
        patch.set_x(patch.get_x() + (width - new_width) / 2)
        patch.set_width(new_width)


def render_metric_group(summary_df: pd.DataFrame, metric: str, metric_label: str, models: list[str], out_path: Path, title: str) -> None:
    work = summary_df[summary_df['model_name'].isin(models)].copy()
    fig, ax = plt.subplots(figsize=(5.8, 4.4), dpi=220)
    if work.empty or f'mean_{metric}' not in work.columns:
        ax.text(0.5, 0.5, f'No rows for {metric}', ha='center', va='center')
        ax.axis('off')
    else:
        work['dataset_label'] = pd.Categorical(work['dataset_label'], DATASET_ORDER, ordered=True)
        work['plot_label'] = work['model_name'].map(MODEL_LABELS)
        work = work.sort_values(['dataset_label'])
        hue_order = [MODEL_LABELS[m] for m in models]
        palette = {MODEL_LABELS[m]: MODEL_COLORS[MODEL_LABELS[m]] for m in models}
        sns.barplot(data=work, x='dataset_label', y=f'mean_{metric}', hue='plot_label', hue_order=hue_order, palette=palette, ax=ax, saturation=0.95, errorbar=None)
        _shrink_bars(ax, scale=0.9)
        vals = work[f'mean_{metric}'].astype(float)
        ymin = float(vals.min())
        ymax = float(vals.max())
        span = max(ymax - ymin, max(abs(ymax), abs(ymin), 1e-6))
        lower = min(0.0, ymin - 0.08 * span)
        upper = ymax + 0.28 * span
        ax.set_ylim(lower, upper)
        ax.set_title(title, fontsize=10, pad=8)
        ax.set_xlabel('')
        ax.set_ylabel(metric_label)
        ax.tick_params(axis='x', rotation=32)
        for patch in ax.patches:
            patch.set_edgecolor('black')
            patch.set_linewidth(0.5)
        _style_axis(ax)
        legend = ax.get_legend()
        if legend is not None:
            handles = legend.legend_handles
            labels = [t.get_text() for t in legend.get_texts()]
            legend.remove()
            legend = ax.legend(handles, labels, title='', frameon=False, ncol=3, loc='upper left', bbox_to_anchor=(0.02, 0.995), borderaxespad=0.0, handlelength=0.9, handletextpad=0.35, columnspacing=0.8)
            for txt in legend.get_texts():
                txt.set_fontsize(9)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches='tight')
    plt.close(fig)


def render_main_legend(out_path: Path) -> None:
    legend_order = ['TriShift', 'GEARS', 'biolord', 'Perturbed mean', 'scGPT', 'GenePert']
    handles = [plt.Rectangle((0, 0), 1, 1, facecolor=MODEL_COLORS[label], edgecolor='#444444', linewidth=0.7) for label in legend_order]
    fig, ax = plt.subplots(figsize=(8.6, 1.2), dpi=220)
    ax.axis('off')
    ax.legend(handles, legend_order, ncol=3, loc='center', frameon=False, fontsize=10, handlelength=1.0, handletextpad=0.4, columnspacing=1.4)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches='tight', transparent=True)
    plt.close(fig)


def render_single_metric_heatmap(summary_df: pd.DataFrame, metric: str, title: str, cmap: str, out_path: Path) -> None:
    model_order = [MODEL_LABELS[m] for m in EXTRA_HEATMAP_MODELS]
    work = summary_df[summary_df['model_name'].isin(EXTRA_HEATMAP_MODELS)].copy()
    fig, ax = plt.subplots(figsize=(4.6, 4.8), dpi=220)
    col = f'mean_{metric}'
    if work.empty or col not in work.columns:
        ax.text(0.5, 0.5, f'No rows for {metric}', ha='center', va='center')
        ax.axis('off')
    else:
        work['dataset_label'] = pd.Categorical(work['dataset_label'], DATASET_ORDER, ordered=True)
        work['plot_label'] = work['model_name'].map(MODEL_LABELS)
        pivot = work.pivot_table(index='plot_label', columns='dataset_label', values=col, aggfunc='mean').reindex(index=model_order, columns=DATASET_ORDER)
        pivot = pivot.loc[pivot.notna().any(axis=1)]
        annot = pivot.copy().astype(object)
        for r in pivot.index:
            for c in pivot.columns:
                val = pivot.loc[r, c]
                annot.loc[r, c] = 'NA' if pd.isna(val) else f'{float(val):.3f}'
        ax.set_facecolor('#ECECEC')
        if pivot.shape[0] == 0 or not pivot.notna().to_numpy().any():
            ax.text(0.5, 0.5, 'No available values', ha='center', va='center')
            ax.axis('off')
        else:
            sns.heatmap(pivot, ax=ax, cmap=cmap, annot=annot, fmt='', linewidths=0.8, linecolor='white', cbar=True, mask=pivot.isna(), square=False)
            ax.set_title(title, fontsize=10)
            ax.set_xlabel('')
            ax.set_ylabel('')
            ax.tick_params(axis='x', rotation=18)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches='tight')
    plt.close(fig)


@lru_cache(maxsize=None)
def _repo_path(path_like):
    path = Path(path_like)
    return path if path.is_absolute() else repo_root / path

def load_dataset_context(dataset_key: str):
    adata = load_adata(str(_repo_path(PATHS_CFG['datasets'][dataset_key])))
    cond_values = adata.obs['condition'].astype(str).map(normalize_condition).to_numpy()
    if 'gene_name' in adata.var.columns:
        gene_names = adata.var['gene_name'].astype(str).tolist()
    else:
        gene_names = adata.var_names.astype(str).tolist()
    gene_index = {gene_name: idx for idx, gene_name in enumerate(gene_names)}
    return adata, cond_values, gene_index


@lru_cache(maxsize=None)
def load_normalized_payload(dataset_key: str, model_name: str, split_id: int):
    _, payload = load_payload_item(dataset=dataset_key, model_name=model_name, split_id=int(split_id), condition=None)
    return {normalize_condition(str(k)): v for k, v in payload.items() if isinstance(v, dict)}


def resolve_case_item(dataset_key: str, condition_key: str, model_name: str, split_policy: str = 'first_available') -> dict[str, object]:
    if split_policy != 'first_available':
        raise ValueError(f'Unsupported split_policy: {split_policy}')
    errors: list[str] = []
    for split_id in SPLIT_IDS:
        try:
            payload = load_normalized_payload(dataset_key, model_name, int(split_id))
        except Exception as exc:
            errors.append(f'split {split_id}: {exc}')
            continue
        item = payload.get(condition_key)
        if item is not None:
            return {'model_name': model_name, 'status': 'ok', 'split_id': int(split_id), 'item': item, 'error': ''}
        errors.append(f'split {split_id}: condition missing')
    return {'model_name': model_name, 'status': 'missing', 'split_id': None, 'item': None, 'error': ' | '.join(errors[:4])}


def mean_vector(x) -> np.ndarray:
    return np.asarray(np.asarray(x, dtype=float).mean(axis=0)).reshape(-1)


def build_truth_reference(dataset_key: str, condition_key: str, reference_genes: list[str]) -> tuple[pd.Series, pd.Series]:
    adata, cond_values, gene_index = load_dataset_context(dataset_key)
    selected_genes = [gene_name for gene_name in reference_genes if gene_name in gene_index]
    if not selected_genes:
        raise ValueError(f'No reference genes could be aligned to raw dataset genes for {dataset_key}')
    idx = np.asarray([gene_index[gene_name] for gene_name in selected_genes], dtype=int)
    ctrl_mask = cond_values == 'ctrl'
    target_mask = cond_values == condition_key
    if int(target_mask.sum()) == 0:
        raise ValueError(f'Condition not found in raw data: {condition_key}')
    ctrl_mean = np.asarray(adata.X[ctrl_mask][:, idx].mean(axis=0)).reshape(-1).astype(float, copy=False)
    target_mean = np.asarray(adata.X[target_mask][:, idx].mean(axis=0)).reshape(-1).astype(float, copy=False)
    truth_delta = target_mean - ctrl_mean
    return pd.Series(truth_delta, index=selected_genes), pd.Series(ctrl_mean, index=selected_genes)


def extract_prediction_series(item: dict[str, object]) -> pd.Series:
    if item.get('Pred_full') is not None and item.get('gene_name_full') is not None:
        pred_mean = mean_vector(item['Pred_full'])
        pred_genes = np.asarray(item['gene_name_full']).astype(str)
    else:
        pred_mean = mean_vector(item['Pred'])
        pred_genes = np.asarray(item.get('DE_name', [f'g{i}' for i in range(pred_mean.shape[0])])).astype(str)
    return pd.Series(pred_mean, index=pred_genes).groupby(level=0).mean()


def safe_corr(a: np.ndarray, b: np.ndarray) -> float:
    x = np.asarray(a, dtype=float).reshape(-1)
    y = np.asarray(b, dtype=float).reshape(-1)
    if x.size == 0 or y.size == 0:
        return float('nan')
    if np.allclose(np.std(x), 0.0) or np.allclose(np.std(y), 0.0):
        return float('nan')
    return float(np.corrcoef(x, y)[0, 1])


def sign_agreement(truth: np.ndarray, pred: np.ndarray) -> float:
    truth_sign = np.sign(np.asarray(truth, dtype=float).reshape(-1))
    pred_sign = np.sign(np.asarray(pred, dtype=float).reshape(-1))
    if truth_sign.size == 0:
        return float('nan')
    return float(np.mean(truth_sign == pred_sign))


def _evaluate_case_condition(dataset_key: str, condition_key: str, display_models: list[str], check_models: list[str], split_policy: str, top_k: int):
    resolved = [resolve_case_item(dataset_key, condition_key, model_name, split_policy=split_policy) for model_name in check_models]
    resolved_df = pd.DataFrame([
        {
            'dataset': dataset_key,
            'condition': condition_key,
            'model_name': row['model_name'],
            'label': MODEL_LABELS.get(row['model_name'], row['model_name']),
            'status': row['status'],
            'split_id': row['split_id'],
            'error': row['error'],
        }
        for row in resolved
    ])
    resolved_ok = [row for row in resolved if row['status'] == 'ok' and row['model_name'] in display_models]
    if len(resolved_ok) < len(display_models):
        return None

    reference_item = resolved_ok[0]['item']
    reference_series = extract_prediction_series(reference_item)
    truth_series, ctrl_series = build_truth_reference(dataset_key, condition_key, reference_series.index.astype(str).tolist())

    pred_abs_series: dict[str, pd.Series] = {}
    common_gene_set = set(truth_series.index.astype(str))
    for row in resolved_ok:
        label = MODEL_LABELS.get(row['model_name'], row['model_name'])
        pred_series = extract_prediction_series(row['item'])
        pred_abs_series[label] = pred_series
        common_gene_set &= set(pred_series.index.astype(str))

    if len(common_gene_set) < top_k:
        return None

    common_gene_index = pd.Index(sorted(common_gene_set))
    truth_common = truth_series.reindex(common_gene_index).astype(float)
    ordered_genes = truth_common.abs().sort_values(ascending=False).index[:top_k].tolist()

    metrics_rows = []
    plot_rows = [pd.DataFrame({'Gene': ordered_genes, 'Expression': truth_series.reindex(ordered_genes).astype(float).to_numpy(), 'Group': 'Truth'})]
    for row in resolved_ok:
        label = MODEL_LABELS.get(row['model_name'], row['model_name'])
        pred_abs = pred_abs_series[label].reindex(ordered_genes).astype(float)
        pred_delta = pred_abs - ctrl_series.reindex(ordered_genes).astype(float)
        truth_sub = truth_series.reindex(ordered_genes).astype(float)
        diff = (pred_delta - truth_sub).abs()
        metrics_rows.append({
            'dataset': dataset_key,
            'condition': condition_key,
            'model_name': row['model_name'],
            'label': label,
            'split_id': row['split_id'],
            'pearson_to_truth': safe_corr(truth_sub.to_numpy(), pred_delta.to_numpy()),
            'mae_to_truth': float(diff.mean()) if len(diff) else float('nan'),
            'sign_agreement': sign_agreement(truth_sub.to_numpy(), pred_delta.to_numpy()),
        })
        plot_rows.append(pd.DataFrame({'Gene': ordered_genes, 'Expression': pred_delta.to_numpy(), 'Group': label}))

    plot_df = pd.concat(plot_rows, ignore_index=True)
    metrics_df = pd.DataFrame(metrics_rows).sort_values(['sign_agreement', 'pearson_to_truth', 'mae_to_truth'], ascending=[False, False, True]).reset_index(drop=True)

    tri_row = metrics_df.loc[metrics_df['model_name'] == 'trishift_nearest'].iloc[0]
    base_rows = metrics_df.loc[metrics_df['model_name'] != 'trishift_nearest']
    best_base_sign = float(base_rows['sign_agreement'].max())
    best_base_corr = float(base_rows['pearson_to_truth'].max())
    best_base_mae = float(base_rows['mae_to_truth'].min())
    meta = {
        'dataset': dataset_key,
        'condition': condition_key,
        'top_k': top_k,
        'ordered_genes': ordered_genes,
        'selected_splits': {row['model_name']: row['split_id'] for row in resolved},
        'tri_sign_gain': float(tri_row['sign_agreement']) - best_base_sign,
        'tri_corr_gain': float(tri_row['pearson_to_truth']) - best_base_corr,
        'tri_mae_gain': best_base_mae - float(tri_row['mae_to_truth']),
        'truth_abs_mean': float(truth_series.reindex(ordered_genes).abs().mean()),
    }
    return meta, plot_df, metrics_df, resolved_df


def select_shared_baseline_case(case_cfg: dict[str, object]):
    dataset_key = str(case_cfg['dataset']).strip()
    display_models = list(case_cfg['display_models'])
    check_models = list(case_cfg.get('check_models', display_models))
    split_policy = str(case_cfg.get('split_policy', 'first_available')).strip()
    top_k = int(case_cfg.get('top_k', 12))
    selection_policy = str(case_cfg.get('selection_policy', 'display_gene_expression_advantage')).strip()
    condition_value = case_cfg.get('condition')
    if condition_value not in (None, '', 'auto'):
        evaluated = _evaluate_case_condition(dataset_key, normalize_condition(str(condition_value).strip()), display_models, check_models, split_policy, top_k)
        if evaluated is None:
            raise ValueError(f'Case condition could not be resolved: {condition_value}')
        meta, plot_df, metrics_df, resolved_df = evaluated
        score_df = pd.DataFrame([{**meta, 'selection_policy': selection_policy, 'top_genes': '|'.join(meta['ordered_genes'])}])
        return meta, plot_df, metrics_df, resolved_df, score_df

    shared = None
    for model_name in check_models:
        model_conditions = set()
        for split_id in SPLIT_IDS:
            try:
                payload = load_normalized_payload(dataset_key, model_name, int(split_id))
            except Exception:
                continue
            model_conditions.update(payload.keys())
        shared = model_conditions if shared is None else shared & model_conditions
    candidates = sorted(cond for cond in (shared or set()) if cond != 'ctrl')

    scored = []
    best_bundle = None
    best_key = None
    for condition_key in candidates:
        bundle = _evaluate_case_condition(dataset_key, condition_key, display_models, check_models, split_policy, top_k)
        if bundle is None:
            continue
        meta, _, _, _ = bundle
        rank_key = (meta['tri_sign_gain'], meta['tri_corr_gain'], meta['tri_mae_gain'], meta['truth_abs_mean'])
        scored.append({
            'dataset': dataset_key,
            'condition': condition_key,
            'tri_sign_gain': meta['tri_sign_gain'],
            'tri_corr_gain': meta['tri_corr_gain'],
            'tri_mae_gain': meta['tri_mae_gain'],
            'truth_abs_mean': meta['truth_abs_mean'],
            'top_genes': '|'.join(meta['ordered_genes']),
            'selection_policy': selection_policy,
        })
        if best_key is None or rank_key > best_key:
            best_key = rank_key
            best_bundle = bundle
    if best_bundle is None:
        raise ValueError(f'No shared-baseline case could be resolved for dataset {dataset_key}')
    case_score_df = pd.DataFrame(scored).sort_values(['tri_sign_gain', 'tri_corr_gain', 'tri_mae_gain', 'truth_abs_mean'], ascending=[False, False, False, False]).reset_index(drop=True)
    return (*best_bundle, case_score_df)


def build_case_bundle(case_cfg: dict[str, object]) -> tuple[dict[str, object], pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    meta, plot_df, metrics_df, resolved_df, score_df = select_shared_baseline_case(case_cfg)
    return meta, plot_df, metrics_df, resolved_df, score_df


def render_case_plot(plot_df: pd.DataFrame, title: str, out_path: Path) -> None:
    plt.figure(figsize=(13.2, 5.2), dpi=220)
    ax = sns.barplot(data=plot_df, x='Gene', y='Expression', hue='Group', palette=MODEL_COLORS, errorbar=None, saturation=0.95)
    for patch in ax.patches:
        patch.set_edgecolor('white')
        patch.set_linewidth(0.7)
    ax.set_xlabel('')
    ax.set_ylabel('Expression change over control')
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=32)
    ax.legend(title='', frameon=False, ncol=min(3, plot_df['Group'].nunique()), loc='upper center', bbox_to_anchor=(0.5, 1.18))
    ax.axhline(y=0, color='#4A4A4A', linewidth=0.8, linestyle='-')
    _style_axis(ax)
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches='tight')
    plt.close()


In [4]:
runs = []
for spec in DATASET_SPECS:
    benchmark_run = safe_run_baseline(spec['dataset_key'], MAIN_MODELS, 'benchmark', subgroup_filter=spec.get('benchmark_subgroup_filter'))
    runs.append({**spec, 'benchmark_status': benchmark_run['status'], 'benchmark_run': benchmark_run})

availability_df = pd.DataFrame([{
    'dataset_key': row['dataset_key'],
    'dataset_label': row['dataset_label'],
    'benchmark_status': row['benchmark_status'],
    'benchmark_error': row['benchmark_run'].get('error', ''),
} for row in runs])
availability_df.to_csv(OUT_ROOT / 'dataset_availability.csv', index=False, encoding='utf-8-sig')
display(availability_df)

benchmark_frames = []
for row in runs:
    if row['benchmark_status'] == 'ready':
        df = row['benchmark_run']['result']['summary_df'].copy(); df['dataset_label'] = row['dataset_label']; benchmark_frames.append(df)
benchmark_summary_df = pd.concat(benchmark_frames, ignore_index=True) if benchmark_frames else pd.DataFrame()
benchmark_summary_df.to_csv(OUT_ROOT / 'benchmark_summary_all.csv', index=False, encoding='utf-8-sig')
display(benchmark_summary_df.head())

render_metric_group(benchmark_summary_df, 'pearson', 'Pearson', MAIN_MODELS, OUT_ROOT / 'fig2a_mean_pearson_all_models.png', 'Cross-dataset benchmark | mean Pearson')
render_metric_group(benchmark_summary_df, 'nmse', 'nMSE', MAIN_MODELS, OUT_ROOT / 'fig2b_mean_nmse_all_models.png', 'Cross-dataset benchmark | mean nMSE')
render_metric_group(benchmark_summary_df, 'deg_mean_r2', 'DEG mean $R^2$', MAIN_MODELS, OUT_ROOT / 'fig2c_mean_deg_mean_r2_all_models.png', 'Cross-dataset benchmark | mean DEG $R^2$')
render_metric_group(benchmark_summary_df, 'systema_corr_20de_allpert', 'Systema Pearson', MAIN_MODELS, OUT_ROOT / 'fig2d_mean_systema_corr_all_models.png', 'Cross-dataset benchmark | mean Systema Pearson')
render_main_legend(OUT_ROOT / 'fig2_legend_main.png')


,dataset_key,dataset_label,benchmark_status,benchmark_error
0,adamson,Adamson,ready,
1,norman,Norman,ready,
2,dixit,Dixit,ready,


,dataset,model_name,label,n_rows,split_ids_used,metrics_path,mean_pearson,mean_nmse,mean_deg_mean_r2,mean_systema_corr_20de_allpert,mean_systema_corr_deg_r2,mean_scpram_r2_degs_mean_mean,mean_scpram_r2_degs_var_mean,mean_scpram_wasserstein_degs_sum,dataset_label
0,adamson,trishift_nearest,TriShift nearest,85,"1,2,3,4,5",E:\CODE\trishift\artifacts\results\adamson\met...,0.944659,0.168323,0.805440,0.526377,-0.040505,0.945778,0.825881,3.821235,Adamson
1,adamson,gears,GEARS,80,"1,2,3,4,5",E:\CODE\trishift\artifacts\results\gears\adams...,0.807479,0.319527,0.639048,-0.066592,-0.640349,0.931970,NaN,7.991537,Adamson
2,adamson,biolord,biolord,85,"1,2,3,4,5",E:\CODE\trishift\artifacts\results\biolord\ada...,0.584768,0.920030,-0.095814,-0.620915,-9.780418,0.789314,NaN,10.747268,Adamson
3,adamson,genepert,GenePert,85,"1,2,3,4,5",E:\CODE\trishift\artifacts\results\genepert\ad...,0.923959,0.203959,0.770022,0.427485,-0.015094,0.932140,NaN,7.577101,Adamson
4,adamson,scgpt,scGPT,85,"1,2,3,4,5",E:\CODE\trishift\artifacts\results\scgpt\adams...,0.896241,0.277881,0.682242,-0.022928,-0.748371,0.919543,0.121505,7.299978,Adamson


In [5]:
case_meta, case_plot_df, case_metrics_df, case_resolved_df, case_score_df = build_case_bundle(CASE_SHARED_BASELINE)
case_plot_df.to_csv(OUT_ROOT / 'fig2e_case_expression_values.csv', index=False, encoding='utf-8-sig')
case_metrics_df.to_csv(OUT_ROOT / 'fig2e_case_expression_metrics.csv', index=False, encoding='utf-8-sig')
case_resolved_df.to_csv(OUT_ROOT / 'fig2e_case_resolution.csv', index=False, encoding='utf-8-sig')
case_score_df.to_csv(OUT_ROOT / 'fig2e_case_selection_scores.csv', index=False, encoding='utf-8-sig')
title = f"Fig2e: Shared-baseline case | {case_meta['dataset']} | {case_meta['condition']}"
render_case_plot(case_plot_df, title=title, out_path=OUT_ROOT / 'fig2e_case_expression.png')
print(case_meta)
display(case_metrics_df)
display(case_resolved_df)
display(case_score_df.head(10))
display(case_plot_df.head(24))
print(OUT_ROOT)


{'dataset': 'adamson', 'condition': 'CREB1+ctrl', 'top_k': 12, 'ordered_genes': ['SLC25A6', 'CD99', 'DDIT4', 'HSPA5', 'DDIT3', 'HIST1H4C', 'SLC3A2', 'HERPUD1', 'RPS27', 'RPS10', 'CEBPB', 'BTG1'], 'selected_splits': {'trishift_nearest': 2, 'biolord': 2, 'gears': 3, 'genepert': 2, 'scgpt': 2}, 'tri_sign_gain': 0.41666666666666663, 'tri_corr_gain': 0.45724956464992905, 'tri_mae_gain': 0.2274529299315894, 'truth_abs_mean': 0.8659381688727686}


,dataset,condition,model_name,label,split_id,pearson_to_truth,mae_to_truth,sign_agreement
0,adamson,CREB1+ctrl,trishift_nearest,TriShift,2,0.900122,0.536980,1.000000
1,adamson,CREB1+ctrl,genepert,GenePert,2,0.442872,0.764433,0.583333
2,adamson,CREB1+ctrl,biolord,biolord,2,-0.066468,0.868055,0.583333
3,adamson,CREB1+ctrl,gears,GEARS,3,-0.944667,1.176575,0.083333
4,adamson,CREB1+ctrl,scgpt,scGPT,2,-0.750962,1.122704,0.000000


,dataset,condition,model_name,label,status,split_id,error
0,adamson,CREB1+ctrl,trishift_nearest,TriShift,ok,2,
1,adamson,CREB1+ctrl,biolord,biolord,ok,2,
2,adamson,CREB1+ctrl,gears,GEARS,ok,3,
3,adamson,CREB1+ctrl,genepert,GenePert,ok,2,
4,adamson,CREB1+ctrl,scgpt,scGPT,ok,2,


,dataset,condition,tri_sign_gain,tri_corr_gain,tri_mae_gain,truth_abs_mean,top_genes,selection_policy
0,adamson,CREB1+ctrl,0.416667,0.457250,0.227453,0.865938,SLC25A6|CD99|DDIT4|HSPA5|DDIT3|HIST1H4C|SLC3A2...,display_gene_expression_advantage
1,adamson,BHLHE40+ctrl,0.333333,0.969659,0.264695,0.936655,SLC25A6|CHI3L2|CD99|DDIT4|HSPA5|HIST1H4C|SLC3A...,display_gene_expression_advantage
2,adamson,TMEM167A+ctrl,0.083333,-0.073395,0.046708,0.637316,PDIA3|TMEM167A|SLC3A2|MAT2A|PDIA6|CLCA1|SH3BGR...,display_gene_expression_advantage
3,adamson,YIPF5+ctrl,0.000000,0.042715,0.030404,0.627624,PDIA3|MAT2A|CALR|PDIA6|SLC3A2|HSP90B1|HSPA5|VI...,display_gene_expression_advantage
4,adamson,AARS+ctrl,0.000000,0.016378,0.049837,1.031749,HBZ|HIST1H2BK|HBA1|SH3BGRL3|IFITM2|MDK|MS4A4A|...,display_gene_expression_advantage
5,adamson,SCYL1+ctrl,0.000000,0.010789,-0.015811,0.606214,CALR|PDIA3|MAT2A|HSPA5|TIMP1|SH3BGRL3|INSIG1|A...,display_gene_expression_advantage
6,adamson,EIF2B2+ctrl,0.000000,0.006924,0.429475,0.934148,HBZ|HBA1|PRSS57|CLCA1|MDK|PHGDH|PRSS1|HBG2|HIS...,display_gene_expression_advantage
7,adamson,SYVN1+ctrl,0.000000,0.004743,0.022600,0.580850,HSPA5|MAT2A|VIM|DDIT4|PDIA3|CITED2|SH3BGRL3|RP...,display_gene_expression_advantage
8,adamson,STT3A+ctrl,0.000000,0.004147,0.051843,0.534441,MAT2A|VIM|CITED2|DDIT4|RPS29|APOE|SH3BGRL3|RPS...,display_gene_expression_advantage
9,adamson,EIF2B4+ctrl,0.000000,0.004043,0.033772,0.842806,HBZ|PRSS57|HBA1|CLCA1|MDK|SH3BGRL3|PRSS1|PHGDH...,display_gene_expression_advantage


,Gene,Expression,Group
0,SLC25A6,-2.081289,Truth
1,CD99,-0.915405,Truth
2,DDIT4,-0.887421,Truth
3,HSPA5,-0.816138,Truth
4,DDIT3,-0.782032,Truth
5,HIST1H4C,0.746012,Truth
6,SLC3A2,-0.734188,Truth
7,HERPUD1,-0.721660,Truth
8,RPS27,0.716247,Truth
9,RPS10,0.697154,Truth


E:\CODE\trishift\artifacts\paper_figures\main\Fig2_MultiDatasetBenchmark
